# Run and evaluate LIM on ORAS5

In [ ]:
from omegaconf import OmegaConf
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from importlib import reload

from hyblim.model import lim
from hyblim.data import preproc, eof 

plt.style.use("../../paper.mplstyle")

# Load config 
config = OmegaConf.load("config.yaml")


## Load data

In [ ]:
def load_data_lim(cfg_dataloader: dict) -> list:
    """Load dataset for training LIM.

    Args:
        cfg_dataloader (dict): Configuration for the dataloader.

    Returns:
        dataset (dict): Dictionary containing the training, validation, and test data.
        combined_eof (eof.CombinedEOF): Combined EOF object.
        normalizer_pca (preproc.Normalizer): Normalizer object for the PCs.
    """
    lsm = xr.open_dataset(cfg_dataloader['lsm'])['lsm'] if 'lsm' in cfg_dataloader else None
    
    # Load and preprocess data
    da_arr, eofa_lst = [], []
    for var, cfg_var in cfg_dataloader['variables'].items():
        print(f"Load {var} ...!", flush=True)
        da = xr.open_dataset(cfg_var['path'])[var]

        # Apply land sea mask
        if lsm is not None:
            da = da.where(lsm!=1, other=np.nan)

        # Normalize data 
        if 'normalizer' in cfg_var['preprocess']:
            print("Normalize data ...", flush=True)
            normalizer = preproc.Normalizer(method=cfg_var['preprocess']['normalizer'])
            da = normalizer.fit_transform(da)
            # Store normalizer as an attribute in the Dataarray for the inverse transformation
            da.attrs = normalizer.to_dict()
        da_arr.append(da)

        # PCA
        print("Create PCA ...", flush=True)
        n_components = cfg_var['preprocess']['pca']['n_components']
        eofa = eof.EmpiricalOrthogonalFunctionAnalysis(n_components)

        # Get training interval
        if cfg_dataloader['split']['training'][-1] <= 1: # in percent
            training_interval = (int(cfg_dataloader['split']['training'][0]*len(da['time'])),
                                 int(cfg_dataloader['split']['training'][1]*len(da['time'])))
        else:
            training_interval = cfg_dataloader['split']['training']

        eofa.fit(
            da.isel(time=slice(*training_interval))
        )
        eofa_lst.append(eofa)

    print("Merge variables ...", flush=True)
    gridded_data = xr.merge(da_arr)

    combined_eof = eof.CombinedEOF(eofa_lst, vars=list(gridded_data.data_vars))
    # Normalize PCs
    print("Normalize PCs ...", flush=True)
    z_eof = combined_eof.transform(gridded_data)
    normalizer_pca = preproc.Normalizer(method='zscore')
    eof_data = normalizer_pca.fit_transform(z_eof, dim='time')

    # Split in training and test data
    print("Split in training and test data ...", flush=True)
    n_data_samples = len(eof_data['time'])
    dataset = {}
    for key, interval in cfg_dataloader['split'].items():
        if interval[-1] <= 1: # in percent
            interval = (int(interval[0]*n_data_samples),
                        int(interval[1]*n_data_samples))
        dataset[key] = eof_data.isel(time=slice(*interval))


    return {'dataset': dataset, 'gridded_data': gridded_data, 
            'combined_eof': combined_eof, 'normalizer_pca': normalizer_pca}

In [ ]:
dataloader_cesm2 = load_data_lim(config['dataloader_cesm2'])
dataset_cesm2 = dataloader_cesm2['dataset']


In [ ]:
dataloader_oras5 = load_data_lim(config['dataloader_oras5'])

# Project ORAS5 to CESM2 EOFs
z_oras5 = dataloader_cesm2['combined_eof'].transform(dataloader_oras5['gridded_data'])
z_oras5 = dataloader_oras5['normalizer_pca'].transform(z_oras5, dim='time')


## Create Model

In [ ]:
reload(lim)
start_month = dataset_cesm2['training'].time.dt.month[0].data
model = lim.CSLIM(tau=1)
print("Fit CS-LIM", flush=True)
model.fit(dataset_cesm2['training'].data.T, start_month, average_window=3)
Q = model.noise_covariance()


In [ ]:
reload(preproc)
def hindcast_mean(model, da: xr.DataArray, lag: int) -> xr.DataArray:
    """Hindcast LIM mean for a given lag.

    Args:
        model (_type_): LIM model. 
        da (xr.DataArray): Input dataarray. 
        lag (int, optional): Lag time.

    Returns:
        xr.DataArray: Forecast dataarray
    """
    frcst_arr = []
    for i, t in enumerate(da['time']):
        month = t.dt.month.data
        x_init = da.isel(time=i).data
        x_frcst = model.forecast_mean(x_init, month, lag)
        frcst_arr.append(x_frcst)

    # Use initial times
    da_frcst = xr.DataArray(data=frcst_arr, coords=dict(time=da['time'].data,
                                                       eof=da['eof'].data))
    da_frcst = da_frcst.assign_coords(coords=dict(lag=lag))
    return da_frcst